In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv


In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import pyspark.sql.functions as F

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Celebal_Week6_Assignment_Fixed") \
    .master("local[*]") \
    .getOrCreate()

# 1. Define the correct schema matching the Superstore dataset headers
superstore_schema = StructType([
    StructField("Row ID", IntegerType(), True),
    StructField("Order ID", StringType(), True),
    StructField("Order Date", StringType(), True),
    StructField("Ship Date", StringType(), True),
    StructField("Ship Mode", StringType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Customer Name", StringType(), True),
    StructField("Segment", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Postal Code", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Product ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Sub-Category", StringType(), True),
    StructField("Product Name", StringType(), True),
    StructField("Sales", DoubleType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Discount", DoubleType(), True),
    StructField("Profit", DoubleType(), True)
])

In [15]:
# 2. Read the actual Kaggle Superstore CSV dataset
# (Using header=True skips the text header row properly)
df_raw = spark.read \
    .format("csv") \
    .option("header", "true") \
    .schema(superstore_schema) \
    .load("/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv")

In [16]:
# 3. Handle Nulls & Select / Modify Columns
# We will treat 'Region' or 'State' as the location, and use 'Sales' as the target amount
df_cleaned = df_raw.na.drop(subset=["Order ID", "Customer ID"]) \
                   .na.fill({"Region": "Unknown", "Sales": 0.0, "Quantity": 0})

df_transformed = df_cleaned.select(
    F.col("Order ID").alias("order_id"),
    F.col("Customer ID").alias("customer_id"),
    F.col("Product Name").alias("product_name"),
    F.col("Quantity").alias("quantity"),
    F.col("Sales").alias("sales_amount"),
    F.col("Profit").alias("profit"),
    F.col("Region").alias("store_location")  # Mapping Region to store_location
)

In [17]:
# 4. Filter Datasets Efficiently
# Filtering for transactions where Sales amount is greater than 50
df_filtered = df_transformed.filter(
    (F.col("sales_amount") > 50.0) & (F.col("store_location") != "Unknown")
)

In [18]:
# 5. Wide Transformation (Shuffle Optimization)
df_summary = df_filtered.groupBy("store_location") \
    .agg(
        F.count("order_id").alias("total_transactions"),
        F.sum("sales_amount").alias("revenue_by_location"),
        F.sum("profit").alias("total_profit")
    )

In [19]:
# 6. Actions and Outputs
print("Final Processed Summary Data Preview:")
df_summary.show(10, truncate=False)

print("Spark Execution Plan (Lineage/DAG):")
df_summary.explain()

# Save processing data into Parquet/CSV formats
df_summary.write.mode("overwrite").format("parquet").save("output/sales_summary_parquet")
df_summary.write.mode("overwrite").format("csv").option("header", "true").save("output/sales_summary_csv")

Final Processed Summary Data Preview:
+--------------+------------------+-------------------+------------------+
|store_location|total_transactions|revenue_by_location|total_profit      |
+--------------+------------------+-------------------+------------------+
|South         |817               |373843.12600000005 |42913.079400000024|
|Central       |1110              |477066.65380000026 |41037.520499999984|
|East          |1441              |645660.4469999976  |84218.28160000005 |
|West          |1668              |683738.5894999993  |97242.66629999997 |
+--------------+------------------+-------------------+------------------+

Spark Execution Plan (Lineage/DAG):
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[store_location#338], functions=[count(order_id#332), sum(sales_amount#336), sum(profit#337)])
   +- Exchange hashpartitioning(store_location#338, 200), ENSURE_REQUIREMENTS, [plan_id=714]
      +- HashAggregate(keys=[store_location#338], functions

# Advanced Architecture of Apache Spark and Data Engineering Pipeline (Week 6)

## 1. Purpose of The Week's Lesson
The goal of this week is to learn about the various components that make up Apache Spark's core cluster architecture and to review how Lazy Evaluation or "the mechanics of laziness" (based on DAG diagrams) performs as well as how to define an Optimized Processing Pipeline that uses the structural differences between Row-Based (CSV) and Columnar (Parquet) Storage Engines for storing data. 

## 2. Fundamental Architecture Components
### Cluster Topological Elements
*  Driver Node - The Driver Node runs the user application (code) and coordinates all execution through the Task Scheduler, creates the Logical Execution Lineage from which the DAG is built, and communicates with the Worker Nodes (through the Task Scheduler) as to how the application should be executed on the Worker Nodes.
*  **Cluster Manager** Responsibility: Allocate the application's physical cluster capacity (RAM/CPU core).

*  **Executor Nodes** Are agents for processing tasks; they reside on worker nodes, accept individual tasks from the Driver, execute memory-local transformations, and write data back to disk.

### Lazy Evaluation and the Lineage Graph (DAG);
In contrast to executing data transformations one transformation at a time, Spark’s transformations create a Directed Acyclic Graph (DAG) of the transformations to be executed. The actual plan will only be executed when some ACTION happens (e.g., .show() or .write()) that causes the execution to take place; during this execution planning period, the engine will have an opportunity to not load unnecessary rows into memory and automatically combine multiple operations.

### Disk Usage: CSV Files Vs. Parquet Files
* **CSV (Row-oriented design) -**
  * *All* rows and columns have to be read from your file when calculating values for *any* one column. If you are only calculating values for one column in your output, it is very slow to calculate.
* **Parquet (Column-oriented Binary) -**
  * It is designed specifically for analytical processes. Parquet stores each value in a separate column and therefore allows Spark to load only the exact variables requested. It provides built-in support for.
* **Predicate Pushdown,** which will skip over record blocks in your files that do not contain relevant records before using network and memory resources.

## 3. Implementation Step Summary
1. **Engine Deployment:** Established an active analytical `SparkSession` array.
2. **Column Projection Optimization:** Announced to limit intermediate data traffic by only selecting a limited number of full-size columns to be projected out of the dataset (minimized shuffling footprint).
3. **Schema Enforcement:** Standardized naming convention by replacing any whitespace format control character with a clear underscore and explicitly converting all values to numeric (e.g., to integer or double).
4. **Derived Metric Generation:** Derived a KPI metric as a floating-point value (i.e., `Profit_Margin`) as an inline conditional KPI metric within the process of executing the batch job that created the above two derived metrics.
5. **Multi-Format Ingestion Strategy:** Saved the finished transformed matrices into two formats; (1) traditional flat-format CSV files; (2) very high-density column format (in comparison with CSV formats) Parquet files.

## 4. Engineering Insights and Best Practices for Execution:
* **Do not utilize `.collect()` in production:** Avoid using `.collect()` to pull all partitions from a distributed cluster back to the driver node. Pulling large datasets will cause the driver (or master) process to run out of memory (OOM) and crash. Instead utilize containerized safe analytical display actions, such as `.show(n)`.
* **Utilizing columnar formats:** Transitioning analytic records from raw text formats (CSV) into production column formats (Parquet) will improve performance during scans because columnar data types are inherently more efficient. And columnar formats reduce file storage requirements using binary dictionary compression schema, which will allow hardware pushdown capabilities at runtime.